# DDPM for MNIST -- Evaluation (sampling, trajectory, step-count comparison)

Loads the checkpoint produced by `notebooks/colab/ddpm_mnist_train.ipynb`
and runs everything that does NOT require retraining:

1. **x_T -> x_0 trace**: ancestral sampling with intermediate noise
   levels snapshotted and displayed, pure noise to recognizable digit.
2. **Reverse-step-count comparison**: the SAME trained model, the SAME
   starting noise (seed), sampled with different numbers of reverse
   steps `S` (via `DDPMScheduler.make_respaced_schedule`, the standard
   "respacing" reparameterization -- see
   `code/ddpm_mnist_scheduler.py` for the exact math). Only `S` varies,
   so any visible quality difference is attributable to `S` alone.

All sampling logic lives in `code/ddpm_mnist_sample.py`; this notebook
only calls it and displays the results. Because this is a separate
notebook from training, any new diagnostic can be added here and rerun
in seconds against the existing checkpoint.

## 1. GPU check

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 3. Mount Drive, load the checkpoint

`CHECKPOINT_PATH` must match the path used in the training notebook.
Model and scheduler are reconstructed from the configs saved INSIDE the
checkpoint -- no architecture/schedule hyperparameter needs to be
retyped here.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

CHECKPOINT_PATH = "/content/drive/MyDrive/thesis_data/ddpm_mnist/checkpoint.pt"

from ddpm_mnist_model import MnistUNet, count_trainable_parameters
from ddpm_mnist_scheduler import DDPMScheduler
from ddpm_mnist_train import load_checkpoint

checkpoint = load_checkpoint(CHECKPOINT_PATH, device)
model = MnistUNet(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
scheduler = DDPMScheduler(**checkpoint["scheduler_config"], device=device)

print(f"loaded checkpoint from {CHECKPOINT_PATH}")
print(f"epoch = {checkpoint['epoch']}, trainable parameters = {count_trainable_parameters(model):,}")
print(f"scheduler config = {checkpoint['scheduler_config']}")


## 4. x_T -> x_0 trajectory trace

5 independently generated digits (per this project's results-
visualization convention), with the noise level snapshotted at several
evenly-spaced timesteps along the way.

In [ ]:
from ddpm_mnist_sample import generate_samples, visualize_denoising_trajectory

NUM_SAMPLES = 5
NUM_TRAJECTORY_SNAPSHOTS = 8
SEED = 0

image_shape = (checkpoint["model_config"]["in_channels"], 28, 28)
generator = torch.Generator(device=device).manual_seed(SEED)

track_model_timesteps = set(
    torch.linspace(0, scheduler.num_timesteps - 1, NUM_TRAJECTORY_SNAPSHOTS).round().long().tolist()
)

final_samples, trajectory = generate_samples(
    model, scheduler, NUM_SAMPLES, image_shape, device,
    generator=generator, track_model_timesteps=track_model_timesteps,
)
visualize_denoising_trajectory(trajectory, num_examples=NUM_SAMPLES)


## 5. Comparing different numbers of reverse sampling steps

Same model, same seed (-> same starting `x_T`), varying only `S` (the
number of reverse steps actually taken at sampling time). Fewer steps
means fewer, coarser-grained Gaussian jumps approximating the same
reverse diffusion -- see `print_step_count_discussion`'s printed
explanation of what to expect and why, kept separate from the measured
numbers themselves.

In [ ]:
from ddpm_mnist_sample import compare_num_inference_steps, visualize_step_count_comparison, print_step_count_discussion

STEP_COUNTS = [s for s in [scheduler.num_timesteps, 250, 50, 10] if s <= scheduler.num_timesteps]

results = compare_num_inference_steps(
    model, scheduler, STEP_COUNTS, num_samples=NUM_SAMPLES,
    image_shape=image_shape, device=device, seed=SEED,
)
visualize_step_count_comparison(results)
print_step_count_discussion(results)


## 6. Adding a new diagnostic later

Any new sampling-time check (a different seed, a different `S` list, a
new visualization) only needs a new cell below that calls into
`code/ddpm_mnist_sample.py` -- `model`/`scheduler` above are already
loaded, so nothing here needs to retrain.